<a href="https://colab.research.google.com/github/Fauzi455/PembelajaranMesin_2026/blob/main/JS-09/TUGAS/Tugas%20Lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#TUGAS LAB 1

In [2]:
pip install annoy faiss-cpu hnswlib numpy matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 7.4 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 18.3 MB/s  0:00:01
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520036 sha256=3c4e738e367416ba9693c9ebcfd474a293cd5f0447d754a39e5a90b55193425b
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679799 sha256=79687880b615ed617f1724f8e0168f14e02e256135b23513b780437d8000f828
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50

In [3]:
import time
from annoy import AnnoyIndex
import faiss
import hnswlib
import numpy as np

experiments = [
    {
        "n_samples": 1000,
        "dim": 2,
        "metric_name": "Euclidean (L2)",
        "annoy_metric": "euclidean",
        "hnsw_space": "l2",
        "faiss_metric": "l2",
    },
    {
        "n_samples": 1000,
        "dim": 2,
        "metric_name": "Cosine / Angular",
        "annoy_metric": "angular",
        "hnsw_space": "cosine",
        "faiss_metric": "ip",
    },
    {
        "n_samples": 1000000,
        "dim": 5,
        "metric_name": "Euclidean (L2)",
        "annoy_metric": "euclidean",
        "hnsw_space": "l2",
        "faiss_metric": "l2",
    },
    {
        "n_samples": 1000000,
        "dim": 5,
        "metric_name": "Cosine / Angular",
        "annoy_metric": "angular",
        "hnsw_space": "cosine",
        "faiss_metric": "ip",
    },
]

results = []

for exp in experiments:
  N = exp["n_samples"]
  dim = exp["dim"]

  # Generate dataset dummy secara acak
  np.random.seed(42)
  data = np.random.rand(N, dim).astype("float32")
  query = np.random.rand(1, dim).astype("float32")

  if exp["faiss_metric"] == "ip":
    faiss.normalize_L2(data)
    faiss.normalize_L2(query)

  # --- 1. ANNOY ---
  start = time.time()
  t = AnnoyIndex(dim, exp["annoy_metric"])
  for i, vector in enumerate(data):
    t.add_item(i, vector)
  t.build(10)
  annoy_time = time.time() - start

  # --- 2. FAISS ---
  start = time.time()
  if exp["faiss_metric"] == "l2":
    index = faiss.IndexHNSWFlat(dim, 32)
  else:
    index = faiss.IndexHNSWFlat(dim, 32, faiss.METRIC_INNER_PRODUCT)
  index.add(data)
  faiss_time = time.time() - start

  # --- 3. HNSW (hnswlib) ---
  start = time.time()
  p = hnswlib.Index(space=exp["hnsw_space"], dim=dim)
  p.init_index(max_elements=N, ef_construction=100, M=16)
  p.add_items(data)
  p.set_ef(50)
  hnsw_time = time.time() - start

  results.append({
      "label": f"{N} / {dim}D ({exp['metric_name']})",
      "annoy": f"{annoy_time:.4f} s",
      "faiss": f"{faiss_time:.4f} s",
      "hnsw": f"{hnsw_time:.4f} s",
  })

# Menampilkan Ringkasan Hasil
print(f"{'Jumlah Data/Dimensi & Metric':<35} | {'ANNOY':<10} | {'FAISS':<10} | {'HNSW':<10}")
print("-" * 60)
for r in results:
  print(f"{r['label']:<35} | {r['annoy']:<10} | {r['faiss']:<10} | {r['hnsw']:<10}")
print("=" * 60)

Jumlah Data/Dimensi & Metric        | ANNOY      | FAISS      | HNSW      
------------------------------------------------------------
1000 / 2D (Euclidean (L2))          | 0.0370 s   | 0.0663 s   | 0.0667 s  
1000 / 2D (Cosine / Angular)        | 0.0680 s   | 0.0980 s   | 0.0539 s  
1000000 / 5D (Euclidean (L2))       | 24.1012 s  | 66.5297 s  | 90.9706 s 
1000000 / 5D (Cosine / Angular)     | 28.1558 s  | 60.5505 s  | 84.1219 s 
